# Kaggle runner: preprocess, train, evaluate

**Educational use only. Not a medical device.**

Runs the project pipeline on the full RSNA Pneumonia training data on a Kaggle GPU and
packages the checkpoint and reports for download.

Before running, in the notebook settings:
1. **Add data**: attach the `rsna-pneumonia-detection-challenge` competition.
2. **Accelerator**: GPU.
3. **Internet**: on (needed for `git clone`, pip and the ImageNet weights).

No Kaggle token is needed: the data comes from the attached competition, not the API.
This notebook prints no patient IDs and displays no images.

## 1. Check the GPU

Training DenseNet121 on about 26,700 images is impractical on CPU, so stop early if no
GPU is attached rather than discover it hours later.

In [ ]:
import torch

print("torch", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU attached. Set Accelerator to GPU in the notebook settings.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Find the RSNA data

The competition mount path has differed between Kaggle setups, so the folder holding
`stage_2_train_labels.csv` is found by searching `/kaggle/input` instead of hardcoding it.
Only folder names and file counts are printed.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
print("Found under /kaggle/input:")
for p in sorted(INPUT_ROOT.iterdir()):
    print("  ", p)

matches = sorted(INPUT_ROOT.glob("*/stage_2_train_labels.csv"))
matches += sorted(INPUT_ROOT.glob("*/*/stage_2_train_labels.csv"))
if len(matches) != 1:
    raise RuntimeError(f"Expected one RSNA folder with stage_2_train_labels.csv, found {matches}")

RSNA = matches[0].parent
LABELS = RSNA / "stage_2_train_labels.csv"
IMAGES = RSNA / "stage_2_train_images"
print("RSNA folder:", RSNA)
print("contents:", sorted(p.name for p in RSNA.iterdir()))
print("training DICOMs:", sum(1 for _ in IMAGES.glob("*.dcm")))

## 3. Clone the project and define a command runner

`run()` streams a command's output as it runs and raises if it fails, so a failed step
stops the notebook instead of letting later steps run on missing data. tqdm progress is
throttled to one update every 30 seconds to keep the log readable.

In [ ]:
import os
import subprocess
import sys

WORK = Path("/kaggle/working")
REPO_URL = "https://github.com/ayoitshasya/radiology-imaging-diagnosis.git"
REPO = WORK / "radiology-imaging-diagnosis"
os.environ["TQDM_MININTERVAL"] = "30"


def run(*args, cwd=REPO):
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")


if not REPO.exists():
    run("git", "clone", "--depth", "1", REPO_URL, REPO, cwd=WORK)
run("git", "log", "-1", "--format=commit %H  %s")

## 4. Install only missing packages

Kaggle images already include most dependencies, so only absent ones are installed.
pydicom is also upgraded if older than 3.0, because the pipeline uses
`pydicom.pixels.apply_voi_lut`, which pydicom 2 does not have. torch is never reinstalled,
to keep Kaggle's CUDA build.

In [ ]:
import importlib.util
from importlib.metadata import version

# import name -> pip package
REQUIRED = {
    "pydicom": "pydicom", "pylibjpeg": "pylibjpeg", "libjpeg": "pylibjpeg-libjpeg",
    "cv2": "opencv-python-headless", "sklearn": "scikit-learn", "yaml": "pyyaml",
    "tqdm": "tqdm", "pandas": "pandas", "matplotlib": "matplotlib",
}
to_install = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if "pydicom" not in to_install and int(version("pydicom").split(".")[0]) < 3:
    to_install.append("pydicom>=3")

print("to install:", to_install or "nothing")
if to_install:
    run(sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *to_install, cwd=WORK)
for dist in ("pydicom", "scikit-learn", "opencv-python-headless", "torchvision"):
    print(f"{dist}: {version(dist)}")

## 5. Preprocess the full training set

Runs the same `build_dataset.py` used locally, with size and seed from the repo's
`config.yaml`. PNGs go to `/kaggle/working/processed`; the split summary is written into
the repo's `reports/` folder so it is packaged with the other results.

In [ ]:
import yaml

cfg = yaml.safe_load((REPO / "config.yaml").read_text())
PROCESSED = WORK / "processed"
run(
    sys.executable, "ml/preprocessing/build_dataset.py",
    "--input", IMAGES,
    "--labels", LABELS,
    "--output", PROCESSED,
    "--size", cfg["preprocess"]["size"],
    "--seed", cfg["project"]["seed"],
    "--report", REPO / "reports" / "preprocessing_summary.json",
)

## 6. Train

Settings (epochs, batch size, learning rate, early stopping) come from `config.yaml`.
The best checkpoint by validation AUROC is saved to `models/`.

In [ ]:
run(sys.executable, "-m", "ml.classification.train", "--device", "cuda",
    "--data-dir", PROCESSED)

## 7. Evaluate and run baselines

`evaluate.py` picks the threshold on validation and applies it unchanged to test.
`baselines.py` scores the view-only shortcut and the majority class on the same test
split, so the model's gain over the shortcut can be reported.

In [ ]:
run(sys.executable, "-m", "ml.classification.evaluate", "--device", "cuda",
    "--data-dir", PROCESSED)
run(sys.executable, "-m", "ml.classification.baselines", "--data-dir", PROCESSED)

## 8. Package results

Zips `models/` and `reports/` into `/kaggle/working/results.zip` for download from the
Output tab. The processed PNGs are not included.

In [ ]:
import zipfile

archive = WORK / "results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("models", "reports"):
        for path in sorted((REPO / folder).rglob("*")):
            if path.is_file():
                zf.write(path, path.relative_to(REPO))

with zipfile.ZipFile(archive) as zf:
    for info in zf.infolist():
        print(f"{info.file_size / 1e6:9.2f} MB  {info.filename}")
print(f"\n{archive}: {archive.stat().st_size / 1e6:.1f} MB")